## 7. Broadcasting & Vectorisation

### Broadcasting rules (right-to-left on shapes)
1. Expanding single elements -> Prepend 1s to the shorter shape
2. Matching dimensions -> Dimensions of size 1 stretch to match
3. Incompartible Shapes -> Non-matching sizes that aren't 1 → `ValueError`

### Vectorisation
Replace Python loops with NumPy operations — the loops run in compiled C.

| Tool | Use |
|------|-----|
| Direct operations `a*b` | Fastest - always prefer |
| `np.select` | Multi-condition element-wise choice |
| `np.vectorize` | Wrap scalar function (still Python loop internally) |
| `np.frompyfunc` | Like vectorize but returns object array |

In [17]:
import numpy as np

## Broadcasting

In [18]:
# with loop
prices = [100,200,300]
discount = 10 
final_prices = []
for price in prices:
    final_price = price - (price * discount/100)
    final_prices.append(final_price)
print(final_prices)

# with numpy
prices = np.array([100,200,300])
discount = 10
final_prices = prices - (prices * discount/100)
print(final_prices)

[90.0, 180.0, 270.0]
[ 90. 180. 270.]


In [21]:
# Scalar broadcast

a = np.array([[1,2,3],[4,5,6]])
print('Original array: \n', a)

result = a + 10
print('a + 10: \n', result)

Original array: 
 [[1 2 3]
 [4 5 6]]
a + 10: 
 [[11 12 13]
 [14 15 16]]


In [22]:
# 1D array

arr = np.array([1,2,3])
print('Original array: ', arr)

result = arr * 2
print('arr * 2: ' ,result)

Original array:  [1 2 3]
arr * 2:  [2 4 6]


In [24]:
# 2D array

matrix = np.array([[1,2,3],[4,5,6]])
vector = np.array([10,20,30])

result = matrix + vector
print('Result : \n', result)

Result : 
 [[11 22 33]
 [14 25 36]]


In [29]:
# Row vector broadcast
a = np.array([[1,2,3],[4,5,6]])
print('Original array: \n', a)

row_bias = np.array([10,20,30])
print('a + [10,20,30]:\n', a + row_bias)

Original array: 
 [[1 2 3]
 [4 5 6]]
a + [10,20,30]:
 [[11 22 33]
 [14 25 36]]


In [30]:
# Column vector broadcast
a = np.array([[1,2,3],[4,5,6]])
print('Original array: \n', a)

col_bias = np.array([[100],[200]])
print('a + [[100],[200]]:\n', a + col_bias)

Original array: 
 [[1 2 3]
 [4 5 6]]
a + [[100],[200]]:
 [[101 102 103]
 [204 205 206]]


In [27]:
# Row-wise min-max normalisation

data = np.array([[10.,20.,30.],[100.,200.,300.],[1.,2.,3.]])
row_min = data.min(axis=1,keepdims=True)
row_max = data.max(axis=1,keepdims=True)
norm    = (data - row_min) / (row_max - row_min)
print('Row-normalised:\n', norm)

Row-normalised:
 [[0.  0.5 1. ]
 [0.  0.5 1. ]
 [0.  0.5 1. ]]


In [31]:
# error

arr1 = np.array([[1,2,3],[4,5,6]])
arr2 = np.array([1,2])

result = arr1 + arr2 # operands could not be broadcast together with shapes (2,3) (2,) 

print(result) 

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 

## Vectorisation

In [32]:
# with list comprehension
list1 = [1,2,3]
list2 = [4,5,6]
result = [x+y for x,y in zip(list1, list2)]
print(result)

# with numpy - very fast
arr1 = np.array([1,2,3,4,5])
arr2 = np.array([1,2,3,4,5])
result = arr1 + arr2 
print(result)

[5, 7, 9]
[ 2  4  6  8 10]


In [34]:
# Vectorisation speed
import time
SIZE = 500_000
rng  = np.random.default_rng(0)
arr  = rng.random(SIZE)

t0 = time.perf_counter()
loop_res = [float(x**2+2*x+1) for x in arr]
loop_time = time.perf_counter() - t0

t0 = time.perf_counter()
vec_res = arr**2 + 2*arr + 1
vec_time = time.perf_counter() - t0

print(f'Loop time  : {loop_time:.4f}s')
print(f'Vectorised : {vec_time:.4f}s')
print(f'Speedup    : {loop_time/vec_time:.1f}x')

Loop time  : 0.4163s
Vectorised : 0.0105s
Speedup    : 39.5x


In [16]:
arr = np.array([10,20,30,40,50])
multiplied = arr * 2

print(multiplied)

[ 20  40  60  80 100]


In [35]:
# np.vectorize
def classify_score(s):
    if s>=90: return 'A'
    elif s>=75: return 'B'
    elif s>=60: return 'C'
    elif s>=40: return 'D'
    return 'F'

vclassify = np.vectorize(classify_score)
scores = np.array([95,82,61,38,75,90,55,88,42,70])
print('Scores :', scores)
print('Grades :', vclassify(scores))

# Same with np.select (truly vectorised)
conds  = [scores>=90, scores>=75, scores>=60, scores>=40]
choice = ['A','B','C','D']
print('np.select:', np.select(conds, choice, default='F'))

Scores : [95 82 61 38 75 90 55 88 42 70]
Grades : ['A' 'B' 'C' 'F' 'B' 'A' 'D' 'B' 'D' 'C']
np.select: ['A' 'B' 'C' 'F' 'B' 'A' 'D' 'B' 'D' 'C']
